# GulfDealFlow — Script 03: Master Merger
Combines all source files into one clean, deduplicated master dataset.

**Run after:** Script 01 and Script 02 are complete and you've reviewed `news_staging.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pandas -q

In [ ]:
import pandas as pd, os
from datetime import datetime

BASE_DIR    = '/content/drive/MyDrive/GulfDealFlow'
OUTPUT_PATH = os.path.join(BASE_DIR, 'gcc_ventures_master.csv')

SOURCE_FILES = [
    os.path.join(BASE_DIR, 'gcc_ventures_template.csv'),
    os.path.join(BASE_DIR, 'crunchbase_cleaned.csv'),
    os.path.join(BASE_DIR, 'news_staging.csv'),
]

REQUIRED_COLUMNS = [
    'deal_id', 'company_name', 'country', 'city', 'date', 'stage',
    'amount_usd', 'disclosed', 'sector', 'description', 'founded_year',
    'website', 'lead_investor', 'co_investors', 'investor_types', 'source', 'notes'
]

VALID_COUNTRIES = ['UAE', 'Saudi Arabia', 'Kuwait', 'Bahrain', 'Oman', 'Qatar']
VALID_STAGES    = ['Pre-Seed', 'Seed', 'Series A', 'Series B', 'Series C+', 'Growth', 'Undisclosed']
VALID_SECTORS   = [
    'Fintech', 'Proptech', 'Logistics & Supply Chain', 'Healthtech', 'Edtech',
    'E-commerce & Retail', 'SaaS & Enterprise Software', 'Deep Tech & AI',
    'Energy & Cleantech', 'Media & Entertainment', 'Food & Agritech', 'Other'
]

def load_source(path):
    df = pd.read_csv(path, dtype=str).fillna('')
    for col in REQUIRED_COLUMNS:
        if col not in df.columns: df[col] = ''
    return df[REQUIRED_COLUMNS]

def deduplicate(df):
    before = len(df)
    df['_n'] = df['company_name'].str.lower().str.strip()
    df['_d'] = df['date'].str[:7]
    df['_s'] = df['stage'].str.lower().str.strip()
    df = df.drop_duplicates(subset=['_n','_d']).drop_duplicates(subset=['_n','_s'], keep='first')
    df = df.drop(columns=['_n','_d','_s'])
    print(f'  Deduplication: {before} → {len(df)} (removed {before-len(df)})')
    return df

def validate(df):
    issues = 0
    for i, row in df.iterrows():
        flags = []
        if not row['company_name'].strip(): flags.append('MISSING: company_name')
        if row['country'] not in VALID_COUNTRIES: flags.append(f"INVALID COUNTRY: {row['country']}")
        if row['stage'] not in VALID_STAGES: flags.append(f"INVALID STAGE: {row['stage']}")
        if row['sector'] not in VALID_SECTORS: flags.append(f"INVALID SECTOR: {row['sector']}")
        if not row['date']: flags.append('MISSING: date')
        if flags:
            df.at[i, 'notes'] = ' | '.join(flags) + (' | ' + row['notes'] if row['notes'] else '')
            issues += 1
    print(f'  {"⚠ " + str(issues) + " rows flagged" if issues else "✓ All rows valid"}')
    return df

def assign_ids(df):
    df = df.reset_index(drop=True)
    df['deal_id'] = [f'GCC{str(i+1).zfill(4)}' for i in range(len(df))]
    return df

# ── Run ─────────────────────────────────────────────────────────
print('GulfDealFlow — Master Merger\n')
frames = []
for path in SOURCE_FILES:
    if os.path.exists(path):
        print(f'Loading: {os.path.basename(path)}')
        df = load_source(path)
        df = df[df['company_name'].str.strip() != '']
        print(f'  {len(df)} rows')
        frames.append(df)
    else:
        print(f'Skipping (not found): {os.path.basename(path)}')

if not frames:
    print('\n⚠ No source files found. Run Scripts 01 and 02 first.')
else:
    master = pd.concat(frames, ignore_index=True)
    print('\nDeduplicating...'); master = deduplicate(master)
    print('Validating...');     master = validate(master)
    print('Assigning IDs...');  master = assign_ids(master)
    master['_s'] = pd.to_datetime(master['date'], format='%Y-%m', errors='coerce')
    master = master.sort_values('_s', ascending=False).drop(columns=['_s']).reset_index(drop=True)
    master.to_csv(OUTPUT_PATH, index=False)
    print(f'\n✓ Saved: {OUTPUT_PATH}')

    # Summary
    print(f"\n{'='*50}")
    print(f'  GULFDEALFLOW MASTER DATASET')
    print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print(f"{'='*50}")
    print(f'  Total deals: {len(master)}')
    try:
        disc = master[master['disclosed'].str.upper()=='TRUE']
        total = disc['amount_usd'].replace('','0').astype(float).sum()
        print(f'  Disclosed capital: ${total/1e9:.2f}B')
    except: pass
    print(f'\n  BY COUNTRY:\n' + master['country'].value_counts().to_string())
    print(f'\n  BY STAGE:\n'   + master['stage'].value_counts().to_string())
    print(f'\n  BY SECTOR:\n'  + master['sector'].value_counts().to_string())
    print(f'\n  BY SOURCE:\n'  + master['source'].value_counts().to_string())
    print(f"{'='*50}")

In [ ]:
# Preview master dataset
master.head(20)